# KMOS Stacked – Interactive Inspector

Standalone notebook. Set `USE_EXTERNAL = True` and run **insp-load** to scan
`kmos_proc_server_comb/` for DATACUBE_COMBINED files, then run **insp-widget**.

**Click the spectrum** (top-left or zoom) → prints wavelength, rest-frame λ,
nearest known lines.  
**Click the white-light map** (bottom-right) → re-centres the 5×5 aperture.  
Use the **dropdown** to switch galaxy.  

Requires `ipympl` (`pip install ipympl`) and `ipywidgets`.


In [ ]:
import re, csv, io, base64
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display as _display
from astropy.io import fits

USE_EXTERNAL = True
COMB_BASE    = 'kmos_proc_server_comb'
BASE         = 'kmos_ob_proc_server'
SUFFIX       = '_s_i_bkg2d_rv_tell.fits'
APERTURE     = 5
EDGE_MARGIN  = 1
HA_REST      = 0.65628   # µm
Z_CSV        = Path('scripts/target_redshifts.csv')


In [ ]:
# ── redshift table ──────────────────────────────────────────────────────────
_Z_TABLE = {}; _Z_TYPE = {}; _Z_QUALITY = {}
if Z_CSV.exists():
    with open(Z_CSV, newline='') as fh:
        for row in csv.DictReader(fh):
            k = (row.get('field','').strip(), row.get('arm_target_id','').strip())
            if k[0] and k[1]:
                _Z_TABLE[k]   = float(row.get('redshift','0') or 0)
                _Z_TYPE[k]    = row.get('redshift_type','?').strip()
                _Z_QUALITY[k] = row.get('quality','?').strip()

def _get_z(field, tid):
    k = (str(field), str(tid))
    return _Z_TABLE.get(k, 0.0), _Z_TYPE.get(k,'?'), _Z_QUALITY.get(k,'?')

def wavelength_axis(hdr):
    cdelt = hdr.get('CDELT3', hdr.get('CD3_3', 1.0))
    return (np.arange(hdr['NAXIS3']) + 1 - hdr['CRPIX3']) * cdelt + hdr['CRVAL3']

def ifu_center(cube):
    wl = np.nansum(cube, axis=0)
    fin = wl[np.isfinite(wl)]
    med = np.nanmedian(fin) if fin.size else 0
    wl[~np.isfinite(wl)] = 0
    idx = np.argwhere(wl > med)
    if idx.size:
        cy, cx = idx.mean(axis=0).astype(int)
    else:
        cy, cx = cube.shape[1]//2, cube.shape[2]//2
    return int(cy), int(cx)

def _extract_spec(cube, ctr, box=APERTURE):
    cy, cx = ctr; h2 = box // 2
    y0, y1 = max(cy-h2, 0), min(cy+h2+1, cube.shape[1])
    x0, x1 = max(cx-h2, 0), min(cx+h2+1, cube.shape[2])
    patch = cube[:, y0:y1, x0:x1].reshape(cube.shape[0], -1)
    spec  = np.full(cube.shape[0], np.nan)
    valid = np.isfinite(patch)
    ok    = valid.any(axis=1)
    spec[ok] = np.nansum(np.where(valid[ok], patch[ok], np.nan), axis=1)
    ref = np.nanmedian(spec[np.isfinite(spec)])
    if np.isfinite(ref) and ref != 0:
        spec /= ref
    return spec


In [ ]:
# ── scan COMB_BASE for DATACUBE_COMBINED files ──────────────────────────────
_pat = re.compile(
    r'DATACUBE_COMBINED_(?P<field>.+?)_(?P<tid>\d+)_(?P<band>[A-Z]+)\.fits$'
)

all_entries = []
_gid_n = [90000]

for fits_path in sorted(Path(COMB_BASE).rglob('DATACUBE_COMBINED_*.fits')):
    m = _pat.search(fits_path.name)
    if not m:
        continue
    field, tid, band = m.group('field'), m.group('tid'), m.group('band')
    z, z_type, z_quality = _get_z(field, tid)
    with fits.open(fits_path, memmap=False) as hdul:
        hdr  = hdul['DATA'].header
        cube = hdul['DATA'].data.astype(float)
    wave = wavelength_axis(hdr)
    ref  = np.nanmedian(cube[np.isfinite(cube)])
    if np.isfinite(ref) and ref != 0:
        cube /= ref
    ha_obs = HA_REST * (1 + z)
    hi = int(np.argmin(np.abs(wave - ha_obs)))
    p0, p1 = max(hi-5, 0), min(hi+6, cube.shape[0])
    ha_im = np.nansum(cube[p0:p1], axis=0)
    cy, cx = ifu_center(cube)
    spec   = _extract_spec(cube, (cy, cx))
    ha_snr = (np.nanmax(spec[p0:p1]) / np.nanstd(spec[:p0])
              if p0 > 10 else 0.0)
    gid = _gid_n[0]; _gid_n[0] += 1
    all_entries.append(dict(
        group_id=gid, band=band, field=field, tid=tid,
        z=z, z_type=z_type, z_quality=z_quality,
        wave=wave, cube=cube, ha_image=ha_im,
        ctr_ha=(cy, cx), snr_ha=ha_snr,
        fits_path=str(fits_path),
    ))

print(f'Loaded {len(all_entries)} galaxies')
print(f'Z_CSV keys available: {list(_Z_TABLE.keys())[:5]} ...')
for e in all_entries:
    print(
        f'  gid {e["group_id"]:6d}  {e["band"]:4s}  '
        f'field={e["field"]!r}  tid={e["tid"]!r}  '
        f'z={e["z"]:.4f} [{e["z_type"]}]  '
        f'Ha_SNR={e["snr_ha"]:.1f}  {Path(e["fits_path"]).name}'
    )


## Widget

- **Click the spectrum** (top-left or zoom panel) to identify a line.
- **Click the white-light map** (bottom-right) to re-centre the extraction aperture.
- Use the **dropdown** to switch galaxy.


In [ ]:
# ── Inline-backend inspector ─────────────────────────────────────────────────
# No ipympl needed: plot rendered to base64 PNG in a widgets.HTML widget.
%matplotlib inline

_ID_LINES = {
    'Ha':           0.65628, '[NII]6583':  0.65835, '[NII]6548':  0.65480,
    '[SII]6716':   0.67163,  '[SII]6731':  0.67308, '[OI]6300':   0.63000,
    '[OIII]5007':  0.50082,  '[OIII]4959': 0.49594, 'Hb':         0.48627,
    'Hg':          0.43405,  'Hd':         0.41017, '[OII]3727':  0.37270,
    'HeI 5876':    0.58756,  'HeI 6678':   0.66780,
    'NaD 5893':    0.58929,  'MgI 5175':   0.51751,
    '[NeIII]3869': 0.38690,
}
_LINE_COLOR = {
    'Ha':          '#d62728', '[OI]6300':   '#8c564b',
    '[NII]6583':   '#e377c2', '[NII]6548':  '#e377c2',
    '[SII]6716':   '#9467bd', '[SII]6731':  '#9467bd',
    '[OIII]5007':  '#17becf', '[OIII]4959': '#17becf',
    'Hb':          '#1f77b4', 'Hg':         '#1f77b4', 'Hd': '#1f77b4',
    '[OII]3727':   '#2ca02c',
    'HeI 5876':    '#ff7f0e', 'HeI 6678':   '#ff7f0e',
    'NaD 5893':    '#bcbd22', 'MgI 5175':   '#bcbd22',
    '[NeIII]3869': '#7f7f7f',
}

_targets = all_entries

# ── controls ─────────────────────────────────────────────────────────────────
_e0   = _targets[0]
_wmin = float(_e0['wave'].min() * 1e4)
_wmax = float(_e0['wave'].max() * 1e4)
_ha0  = float(HA_REST * (1 + _e0['z']) * 1e4)
_cy0, _cx0 = _e0.get('ctr_ha', (8, 8))

_dd_opts = [
    (f'gid {e["group_id"]} | {e["band"]} | z={e["z"]:.4f} [{e["z_type"]}]'
     f' | Ha SNR={e["snr_ha"]:.1f}', i)
    for i, e in enumerate(_targets)
]
_dd  = widgets.Dropdown(options=_dd_opts, value=0, description='Galaxy:',
                         layout=widgets.Layout(width='580px'))
_wav = widgets.BoundedFloatText(value=_ha0, min=_wmin, max=_wmax, step=0.5,
                                 description='lambda (A):',
                                 layout=widgets.Layout(width='220px'))
_cxw = widgets.BoundedIntText(value=_cx0, min=0, max=15, description='cx:',
                               layout=widgets.Layout(width='120px'))
_cyw = widgets.BoundedIntText(value=_cy0, min=0, max=15, description='cy:',
                               layout=widgets.Layout(width='120px'))
_btn = widgets.Button(description='Reset aperture', button_style='warning',
                       layout=widgets.Layout(width='140px'))
_img  = widgets.HTML(value='<i>Loading...</i>')
_info = widgets.HTML(value='')

# ── draw helpers ─────────────────────────────────────────────────────────────
def _mark_lines(ax, wave, z, wlo=None, whi=None):
    for lname, lrest in _ID_LINES.items():
        lo = lrest * (1 + z)
        if not (wave.min() <= lo <= wave.max()): continue
        if wlo is not None and not (wlo <= lo <= whi): continue
        lc = _LINE_COLOR.get(lname, '#aaa')
        ax.axvline(lo * 1e4, color=lc, lw=0.7, ls='--', alpha=0.65, zorder=1)
        if wlo is not None:
            ax.text(lo * 1e4, 0.97, lname,
                    transform=ax.get_xaxis_transform(),
                    rotation=90, ha='right', va='top', fontsize=7, color=lc,
                    bbox=dict(fc='white', ec='none', alpha=0.7, pad=1))

def _fig_to_html(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=90, bbox_inches='tight')
    plt.close(fig)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;height:auto">'

def _redraw(_=None):
    idx  = _dd.value
    e    = _targets[idx]
    wave = e['wave']; z = e['z']; cube = e.get('cube')
    if cube is None: return
    cy   = int(_cyw.value); cx = int(_cxw.value)
    cw_a = float(_wav.value)          # Angstrom
    cw   = cw_a / 1e4                 # µm
    spec = _extract_spec(cube, (cy, cx))
    wa   = wave * 1e4

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.subplots_adjust(hspace=0.44, wspace=0.32)
    ax_spec, ax_nb, ax_zoom, ax_map = axes.flat

    # full spectrum
    ax_spec.plot(wa, spec, color='#1a6faf', lw=0.8, zorder=2)
    ax_spec.axhline(0, color='#ccc', lw=0.4)
    _mark_lines(ax_spec, wave, z)
    ax_spec.axvline(cw_a, color='#d62728', lw=1.4, alpha=0.9, zorder=4)
    fv = spec[np.isfinite(spec)]
    if fv.size:
        y1p = np.nanpercentile(fv, 1); y99 = np.nanpercentile(fv, 99)
        pad = 0.15 * max(y99 - y1p, 0.1)
        ax_spec.set_ylim(y1p - pad, y99 + pad)
    ax_spec.set_xlim(wa.min(), wa.max())
    ax_spec.set_xlabel('lambda_obs (A)', fontsize=8)
    ax_spec.set_ylabel('flux (norm.)', fontsize=8)
    ax_spec.set_title(
        f'gid {e["group_id"]} | {e["band"]} | z={z:.4f} [{e["z_type"]}] | ctr=({cy},{cx})',
        fontsize=8, pad=4)
    ax_spec.tick_params(labelsize=7)

    # narrowband
    hp  = int(np.argmin(np.abs(wave - cw)))
    p0, p1 = max(hp-3, 0), min(hp+4, cube.shape[0])
    nb_im = np.nansum(cube[p0:p1], axis=0)
    fin   = nb_im[np.isfinite(nb_im)]
    if fin.size:
        ax_nb.imshow(nb_im, origin='lower', aspect='equal', cmap='inferno',
                     vmin=np.nanpercentile(fin, 5), vmax=np.nanpercentile(fin, 97))
    ax_nb.plot(cx, cy, '+', ms=8, mew=1.8, color='white', zorder=5)
    h2 = APERTURE // 2
    ax_nb.add_patch(mpatches.Rectangle((cx-h2-.5, cy-h2-.5), APERTURE, APERTURE,
                    lw=1.2, ec='white', fill=False, zorder=5))
    ax_nb.set_title(f'NB @ {cw_a:.1f} A (+/-3 px)', fontsize=8, pad=4)
    ax_nb.tick_params(labelsize=6)

    # spectral zoom
    wlo = max(cw - 0.015, wave.min()); whi = min(cw + 0.015, wave.max())
    m   = (wave >= wlo) & (wave <= whi)
    if m.any():
        ax_zoom.plot(wa[m], spec[m], color='#1a6faf', lw=1.5, zorder=2)
        ax_zoom.axvline(cw_a, color='#d62728', lw=1.4, ls='--', alpha=0.85, zorder=4)
        _mark_lines(ax_zoom, wave, z, wlo, whi)
        fvz = spec[m]; fvz = fvz[np.isfinite(fvz)]
        if fvz.size:
            ylo = np.nanpercentile(fvz, 2); yhi = np.nanmax(fvz)
            pad = 0.18 * max(yhi - ylo, 0.1)
            ax_zoom.set_ylim(ylo - pad, yhi + pad)
        ax_zoom.set_xlim(wlo * 1e4, whi * 1e4)
        ax_zoom.set_title(f'Zoom: {cw_a:.1f} A (+/-150 A)', fontsize=8, pad=4)
    ax_zoom.set_xlabel('lambda_obs (A)', fontsize=8)
    ax_zoom.set_ylabel('flux (norm.)', fontsize=8)
    ax_zoom.tick_params(labelsize=7)

    # white-light map
    wl_im = np.nansum(cube, axis=0)
    finw  = wl_im[np.isfinite(wl_im)]
    if finw.size:
        ax_map.imshow(wl_im, origin='lower', aspect='equal', cmap='viridis',
                      vmin=np.nanpercentile(finw, 2), vmax=np.nanpercentile(finw, 98))
    ax_map.plot(cx, cy, '+', ms=8, mew=1.8, color='red', zorder=5)
    ax_map.add_patch(mpatches.Rectangle((cx-h2-.5, cy-h2-.5), APERTURE, APERTURE,
                     lw=1.2, ec='red', fill=False, zorder=5))
    ax_map.set_title('White-light  (adjust cx/cy above)', fontsize=8, pad=4)
    ax_map.tick_params(labelsize=6)

    _img.value = _fig_to_html(fig)

    # ── HTML info panels ─────────────────────────────────────────────────────
    CSS = '''
    <style>
    .insp-wrap { font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px;
                 display: flex; gap: 28px; margin-top: 6px; flex-wrap: wrap; }
    .insp-box  { background: #1e1e2e; border: 1px solid #3a3a55;
                 border-radius: 6px; padding: 10px 14px; min-width: 320px; }
    .insp-box h4 { margin: 0 0 6px 0; color: #cdd6f4; font-size: 12px;
                   text-transform: uppercase; letter-spacing: .06em; }
    .insp-box table { border-collapse: collapse; width: 100%; }
    .insp-box th { color: #a6adc8; font-weight: 600; font-size: 11px;
                   text-align: left; padding: 3px 10px 3px 0; border-bottom: 1px solid #3a3a55; }
    .insp-box td { padding: 2px 10px 2px 0; color: #cdd6f4; font-family: monospace; font-size: 12px; }
    .insp-box tr:hover td { background: #2a2a3e; }
    .best-match { font-weight: bold; color: #a6e3a1 !important; }
    .insp-header { color: #89b4fa; font-size: 14px; font-weight: 600;
                   font-family: 'Segoe UI', Arial, sans-serif;
                   margin-bottom: 6px; }
    .insp-sub { color: #a6adc8; font-size: 12px;
                font-family: monospace; margin-bottom: 4px; }
    .dot { display:inline-block; width:8px; height:8px;
           border-radius:50%; margin-right:5px; vertical-align:middle; }
    </style>
    '''

    # Left box: nearest lines to cursor
    rest   = cw / (1 + z)
    nearby = sorted([(abs(lr-rest), ln, lr) for ln, lr in _ID_LINES.items()])[:8]
    click_rows = ''
    for i, (drest, ln, lr) in enumerate(nearby):
        cls = ' class="best-match"' if i == 0 else ''
        lc  = _LINE_COLOR.get(ln, '#888')
        dv  = (rest - lr) / lr * 3e5
        click_rows += (
            f'<tr><td{cls}><span class="dot" style="background:{lc}"></span>{ln}</td>'
            f'<td{cls}>{lr*1e4:.1f}</td>'
            f'<td{cls}>{dv:+.0f}</td>'
            f'<td{cls}>{drest*1e4:.1f}</td></tr>'
        )
    left_box = (
        '<div class="insp-box">'
        f'<h4>Cursor: {cw_a:.2f} Å &rarr; &lambda;<sub>rest</sub> = {rest*1e4:.2f} Å</h4>'
        '<table><tr><th>Line</th><th>&lambda;<sub>rest</sub> (Å)</th>'
        '<th>&Delta;v (km/s)</th><th>&Delta;&lambda; (Å)</th></tr>'
        + click_rows + '</table></div>'
    )

    # Right box: all lines in band at galaxy z
    in_band = [
        (lrest * (1 + z) * 1e4, ln, lrest)
        for ln, lrest in _ID_LINES.items()
        if wave.min() * 1e4 <= lrest * (1 + z) * 1e4 <= wave.max() * 1e4
    ]
    in_band.sort()
    band_rows = ''.join(
        f'<tr><td><span class="dot" style="background:{_LINE_COLOR.get(ln,"#888")}"></span>{ln}</td>'
        f'<td>{lrest*1e4:.1f}</td>'
        f'<td>{lo:.1f}</td></tr>'
        for lo, ln, lrest in in_band
    )
    right_box = (
        '<div class="insp-box">'
        f'<h4>Lines in {e["band"]} band at z = {z:.4f} [{e["z_type"]}]</h4>'
        '<table><tr><th>Line</th><th>&lambda;<sub>rest</sub> (Å)</th>'
        '<th>&lambda;<sub>obs</sub> (Å)</th></tr>'
        + (band_rows if band_rows else '<tr><td colspan="3" style="color:#6c7086">none in range</td></tr>')
        + '</table></div>'
    )

    header = (
        f'<div class="insp-header">'
        f'gid {e["group_id"]} &nbsp;|&nbsp; {e["field"]} &nbsp;|&nbsp; tid {e["tid"]}'
        f'</div>'
        f'<div class="insp-sub">'
        f'band: {e["band"]} &nbsp;&nbsp; '
        f'z = {z:.4f} [{e["z_type"]}] &nbsp;&nbsp; '
        f'H&alpha; SNR = {e["snr_ha"]:.1f} &nbsp;&nbsp; '
        f'aperture = ({cy},{cx})'
        f'</div>'
    )
    _info.value = CSS + header + '<div class="insp-wrap">' + left_box + right_box + '</div>'

# ── callbacks ────────────────────────────────────────────────────────────────
def _on_dd(change):
    e = _targets[change['new']]
    wmin = float(e['wave'].min() * 1e4); wmax = float(e['wave'].max() * 1e4)
    _wav.min = wmin; _wav.max = wmax
    _wav.value = float(HA_REST * (1 + e['z']) * 1e4)
    cy0, cx0 = e.get('ctr_ha', (8, 8))
    _cyw.value = cy0; _cxw.value = cx0
    _redraw()

def _on_reset(_):
    e = _targets[_dd.value]
    cy0, cx0 = e.get('ctr_ha', (8, 8))
    _cyw.value = cy0; _cxw.value = cx0
    _redraw()

_dd.observe(_on_dd, names='value')
_wav.observe(_redraw, names='value')
_cxw.observe(_redraw, names='value')
_cyw.observe(_redraw, names='value')
_btn.on_click(_on_reset)

_controls = widgets.HBox(
    [_dd,
     widgets.VBox([_wav]),
     widgets.VBox([widgets.HBox([_cyw, _cxw]), _btn])],
    layout=widgets.Layout(align_items='flex-end', gap='16px')
)
_display(widgets.VBox([_controls, _img, _info],
                       layout=widgets.Layout(gap='4px')))
_redraw()
